# Student Health Prediction — Model Training

This notebook downloads the Kaggle competition data, trains an XGBoost classifier to predict student health conditions, and exports the model for use in a full-stack web application.

---

In [ ]:
import os
import zipfile
import glob

# The system variables MUST be named exactly 'KAGGLE_USERNAME' and 'KAGGLE_KEY'
os.environ['KAGGLE_USERNAME'] = "naveenekanayake"
os.environ['KAGGLE_KEY'] = "KGAT_c7d2765bf447aa3d374b92061abc334c"

# Install all required packages
!pip install -q kaggle pandas scikit-learn xgboost joblib

# Download the competition data
!kaggle competitions download -c playground-series-s6e7

# Unzip using Python's built-in zipfile (works on Windows, Mac, Linux)
with zipfile.ZipFile('playground-series-s6e7.zip', 'r') as z:
    z.extractall()
print("Download complete! Files in directory:")

# Cross-platform file listing
for f in glob.glob('*.csv'):
    size = os.path.getsize(f)
    print(f"  {f} ({size:,} bytes)")

## Step 2: Train XGBoost Classifier

We load the CSV data, encode categorical features, and train an XGBoost model with categorical support.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Load the training data
train = pd.read_csv('train.csv')
print(f"Dataset shape: {train.shape}")
print(f"Columns: {list(train.columns)}")
print(f"Target distribution:\n{train['health_condition'].value_counts()}")
print()

# Separate features and target
X = train.drop(['id', 'health_condition'], axis=1)
y = train['health_condition']

# Encode categorical features natively for XGBoost
for col in X.select_dtypes(include=['object']).columns:
    print(f"Encoding categorical column: {col}")
    X[col] = X[col].astype('category')

# Encode the target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print()

# Split into train/test for validation
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# Train the XGBoost model
model = XGBClassifier(
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1
)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

## Step 3: Export Model for Web Application

Save the trained model and label encoder as `.pkl` files. These will be used by the Node.js backend (via a Python subprocess) to make predictions on new student data.

In [ ]:
import joblib
import os
import glob

# Save the model and the encoder
joblib.dump(model, 'student_health_xgboost.pkl')
joblib.dump(le, 'label_encoder.pkl')

print("Model and label encoder saved successfully!")
print()

# Cross-platform file listing
for f in glob.glob('*.pkl'):
    size = os.path.getsize(f)
    print(f"  {f} ({size:,} bytes)")

## Bonus: Quick Prediction Test

Run a sample prediction to verify the exported model works correctly.  
*Note: `X_test` is loaded from the training cell above — run cells sequentially.*

In [ ]:
# Load the saved model and encoder to verify they work
loaded_model = joblib.load('student_health_xgboost.pkl')
loaded_le = joblib.load('label_encoder.pkl')

# Take a sample from the test set and make a prediction
sample = X_test[:5]
sample_preds = loaded_model.predict(sample)
sample_probs = loaded_model.predict_proba(sample)

print("Sample Predictions:")
for i, (pred, probs) in enumerate(zip(sample_preds, sample_probs)):
    predicted_class = loaded_le.inverse_transform([pred])[0]
    confidence = max(probs) * 100
    print(f"  Sample {i+1}: {predicted_class} (confidence: {confidence:.1f}%)")

print()
print("✅ Model is ready for integration with the full-stack app!")